Graph Convolutional Networks(GCN) has its message passing in the following form:
\begin{align*}
x_i^{(k)}&=\sum_{j \in N(i)\cup \{i\}} C(W\cdot x_j^{k-1})+b\\
C&=\frac{1}{\sqrt{deg(i)}\sqrt{deg(j)}}
\end{align*}
where $W$ is a learnable matrix and $b$ is a learnable parameter.

In [3]:
%pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.5 MB/s  0:00:00


In [20]:
import torch
from torch import nn
from torch_geometric.nn import MessagePassing
import torch.nn.functional as F

In [11]:
from torch_geometric.utils import add_self_loops, degree
class GCNConv(MessagePassing): # inherit functionality from MessagePassing
    def __init__(self, in_channels, out_channels):
      super().__init__(aggr='add')
      self.lin = nn.Linear(in_channels, out_channels, bias=False)
      self.bias=nn.Parameter(torch.Tensor(out_channels))
      self.reset_parameters() # initialize parameters

    def reset_parameters(self):
      self.lin.reset_parameters()
      self.bias.data.zero_()
    def forward(self, x, edge_index):
      # x: (N, in_channels)
      # edge_index: (2, E)
      edge_index, _ =add_self_loops(edge_index, num_nodes=x.size(0))
      x=self.lin(x)
      row, col = edge_index
      deg = degree(col, x.size(0), dtype=x.dtype) # degree() counts the number of nodes appeared
      deg_inv_sqrt = deg.pow(-0.5)
      norm=deg_inv_sqrt[row] * deg_inv_sqrt[col]
      # propagate update each node's features using its neighbors
      # make norm available to message()
      # self.propagate() will call message()
      return self.propagate(edge_index, x=x, norm=norm)+self.bias

    def message(self, x_j, norm):
      return norm.view(-1, 1) * x_j # apply normalization
class GCN(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, 32)
        self.conv2 = GCNConv(32, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        return self.conv2(x, edge_index)

In [4]:
from torch_geometric.datasets import Planetoid
dataset = Planetoid(root='/tmp/Cora', name='Cora')

Processing...
Done!


In [24]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data=dataset[0].to(device)
num_features=dataset.num_features
num_classes=dataset.num_classes
model=GCN(num_features, num_classes).to(device)
optimizer=torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
model.train()
for epoch in range(2000):
  optimizer.zero_grad()
  out=model(data.x, data.edge_index)
  loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
  loss.backward()
  optimizer.step()
  if epoch % 200 == 0:
    print(f'Epoch: {epoch}, Loss: {loss.item()}')

Epoch: 0, Loss: 1.9471335411071777
Epoch: 200, Loss: 0.009764120914041996
Epoch: 400, Loss: 0.007346377242356539
Epoch: 600, Loss: 0.006878610234707594
Epoch: 800, Loss: 0.006756594404578209
Epoch: 1000, Loss: 0.006709618028253317
Epoch: 1200, Loss: 0.00668207136914134
Epoch: 1400, Loss: 0.006667083129286766
Epoch: 1600, Loss: 0.006655555218458176
Epoch: 1800, Loss: 0.006653440650552511


In [28]:
model.eval()

with torch.no_grad(): # don't track gradient
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (
        pred[data.test_mask] == data.y[data.test_mask]
    ).sum().item()

    acc = correct / data.test_mask.sum().item()

print(f"Accuracy: {acc:.4f}")

Accuracy: 0.8060
